<a href="https://colab.research.google.com/github/txebas/Finance/blob/main/Techstocksstudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tech stocks study
# SEC EDGAR Filing Text Extraction for NLP with Python -- Free, No API Key

Use **edgartools** to extract clean text from SEC filings for NLP and AI analysis -- completely free, no API key or paid subscription required. Get full filing text, individual sections, or structured chunks ready for machine learning pipelines and LLM context windows.

**What you'll learn:**
- Extract full filing text as plain text, markdown, or HTML
- Pull specific sections (business, risk factors, MD&A) by item number
- Get word and character counts for each section
- Search within filings for specific terms
- Access structured chunks for LLM context windows
- Compare text volumes across companies

## Install edgartools

In [ ]:
!pip install -U edgartools

## Setup

The SEC requires all automated tools to identify themselves. Replace the email below with your own -- any valid email works.

In [ ]:
import pandas as pd
from edgar import *

# The SEC requires you to identify yourself (any email works)
set_identity("your.name@example.com")

## Extract Filing Text in 3 Lines

Every SEC filing can be converted to plain text, markdown, or HTML. edgartools handles all the parsing automatically:

## Extract Specific Sections by Item Number

For NLP, you often need specific sections rather than the full filing. Parse a 10-K and access any section by item number:

In [ ]:
tenk = filing.obj()

# Key sections for NLP analysis
business = tenk["1"]        # Item 1: Business description
risks = tenk["1A"]          # Item 1A: Risk factors
mda = tenk["7"]             # Item 7: Management Discussion & Analysis

print(f"{'Section':<20s} {'Words':>10s} {'Characters':>12s}")
print(f"{'-'*20} {'-'*10} {'-'*12}")
for name, text in [("Business", business), ("Risk Factors", risks), ("MD&A", mda)]:
    print(f"{name:<20s} {len(text.split()):>10,} {len(text):>12,}")

print(f"\nBusiness description (first 500 chars):\n")
print(business[:500])

## Get the first 1000 US tech company tickers.

Two methods:
  - Method 1 (recommended): NASDAQ screener API — no key needed, works in most environments
  - Method 2 (fallback):    yfinance sector filtering — slower but very accurate

In [ ]:
"""
Get the first 1000 US tech company tickers.

Two methods:
  - Method 1 (recommended): NASDAQ screener API — no key needed, works in most environments
  - Method 2 (fallback):    yfinance sector filtering — slower but very accurate
"""

import requests
import pandas as pd


# ─────────────────────────────────────────────
# METHOD 1 — NASDAQ Screener API (fastest)
# ─────────────────────────────────────────────
def get_tech_tickers_nasdaq(limit: int = 1000) -> list[str]:
    """
    Pulls US Technology-sector tickers directly from the NASDAQ stock screener.
    No API key required.
    """
    url = "https://api.nasdaq.com/api/screener/stocks"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin": "https://www.nasdaq.com",
        "Referer": "https://www.nasdaq.com/",
    }
    params = {
        "tableonly": "true",
        "limit": limit,
        "offset": 0,
        "sector": "Technology",
        "download": "true",
    }

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    rows = response.json()["data"]["rows"]
    tickers = [row["symbol"] for row in rows][:limit]
    return tickers


# ─────────────────────────────────────────────
# METHOD 2 — yfinance sector filter (accurate, slower)
# ─────────────────────────────────────────────
def get_tech_tickers_yfinance(limit: int = 1000) -> list[str]:
    """
    Downloads all US tickers from a public GitHub list, then filters to
    Technology sector using yfinance. Slow (~1-2 min for 1000 results).
    Run: pip install yfinance requests
    """
    import yfinance as yf
    import time

    # Public list of all US-listed tickers
    url = "https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/all/all_tickers.txt"
    r = requests.get(url, timeout=30)
    all_tickers = r.text.strip().split("\n")

    tech_tickers = []
    for ticker in all_tickers:
        if len(tech_tickers) >= limit:
            break
        try:
            info = yf.Ticker(ticker).info
            if info.get("sector") == "Technology" and info.get("country") == "United States":
                tech_tickers.append(ticker)
                print(f"  [{len(tech_tickers)}] {ticker}")
        except Exception:
            pass
        time.sleep(0.05)  # polite rate limiting

    return tech_tickers


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("Fetching 1000 US tech tickers via NASDAQ screener...")
    try:
        tickers = get_tech_tickers_nasdaq(1000)
        method = "NASDAQ screener"
    except Exception as e:
        print(f"NASDAQ screener failed ({e}), falling back to yfinance...")
        tickers = get_tech_tickers_yfinance(1000)
        method = "yfinance"

    print(f"\n✅ Got {len(tickers)} tickers via {method}")
    print("First 20:", tickers[:20])

    # Save to CSV
    pd.DataFrame({"ticker": tickers}).to_csv("tech_tickers.csv", index=False)
    print(f"\nSaved to tech_tickers.csv")

    # The list itself
    print("\ntickers =", tickers)


## Cleaned tech stocks


In [ ]:
# 2. Lista de empresas tecnológicas
import re
#from tqdm import tqdm
from tqdm.notebook import tqdm



def natural_sort_key(item):
    """
    Generates a key for natural sorting of item strings like 'Item 1', 'Item 1A'.
    It splits the string into a list of number and non-number chunks.
    """
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', item)]

tech_list = ['AAPL','MSFT','NVDA', 'GOOGL', 'AMZN','META']
tech_tickers = tickers
tech_list_final = []

# crear un nuevo listado para meter solo aquellas empresas de las que hay referentes

for i in tqdm(tech_tickers):
    print('*'*40)
    print(f"Processing: {i}")

    # Fetch filings for the ticker
    filings = Company(i).get_filings(form="10-K")

    # Check if any filings were found before accessing index [0]
    if len(filings) > 0:
        tech_list_final.append(i)
        filing = filings[0]
        tenk = filing.obj()
        print(f"Found 10-K filed on: {tenk.filing_date}")

        for item in tenk.items:
            text = tenk[item]
            try:
                words = len(text.split())
            except AttributeError:
                print(f"¡Alerta! El elemento '{item}' no contiene texto (es None).")
                words = 0
        '''
        for item in tenk.items:
            text = tenk[item]
            words = len(text.split())
            # print(f"{item:12s} {words:>8,} words")
        '''
    else:
        print(f"No 10-K filings found for {i}. Skipping...")

In [ ]:
print(len(tech_list_final))

## Save and load stocks

In [ ]:
import pickle

mi_lista = tech_list_final

with open('datos_lista.pkl', 'wb') as archivo:
    pickle.dump(mi_lista, archivo)

print("Lista guardada correctamente.")


In [ ]:
import pickle

with open('datos_lista.pkl', 'rb') as archivo:
    lista_recuperada = pickle.load(archivo)

print(lista_recuperada)


### Comparing Word Counts for Key 10-K Items Across Companies

To visualize the comparative disclosure for important sections, we'll collect the word counts for 'Item 1' (Business), 'Item 1A' (Risk Factors), and 'Item 7' (Management's Discussion & Analysis) for each of the `tech_tickers`.

In [ ]:
import pandas as pd
from IPython.display import display

# Initialize a list to store data for plotting
plot_data = []

# Define the key items we want to compare
#key_items_to_plot = ['Item 1', 'Item 1A', 'Item 7']
key_items_to_plot = sorted(tenk.items, key=natural_sort_key)


# Loop through each technology ticker
# for ticker_symbol in tech_tickers:
for ticker_symbol in tqdm(tech_list_final):
    print(f"Processing {ticker_symbol}...")
    try:
        # Get the latest 10-K filing for the company
        filing = Company(ticker_symbol).get_filings(form="10-K")[0]
        tenk = filing.obj()

        # Prepare a dictionary to hold the word counts for this ticker
        ticker_item_counts = {'Ticker': ticker_symbol}

        # Collect word counts for the key items
        for item_name in key_items_to_plot:
            if item_name in tenk.items:
                text = tenk[item_name]
                words = len(text.split())
                ticker_item_counts[item_name] = words
            else:
                # Handle cases where an item might not be present (e.g., if a 10-K structure differs)
                ticker_item_counts[item_name] = 0 # Or numpy.nan if you prefer

        plot_data.append(ticker_item_counts)

    except Exception as e:
        print(f"Could not process {ticker_symbol}: {e}")

# Convert the collected data into a pandas DataFrame
df_item_comparison = pd.DataFrame(plot_data)

display(df_item_comparison)

Para analizar y visualizar la variabilidad del conteo de palabras de los ítems del 10-K a partir de tu DataFrame `df_item_comparison`, hay que tener en cuenta un factor clave: **el volumen de texto de las secciones difiere enormemente entre sí** (por ejemplo, el *Item 7* suele ser muchísimo más largo que el *Item 1A*).

Por lo tanto, no se puede usar métricas absolutas aisladas (como solo la desviación estándar) porque las secciones más largas siempre parecerán tener "más variabilidad" de forma engañosa.


---

### 0. Una corrección crítica antes de calcular: Ceros vs. `NaN`

En el código actual, si un ítem no existe, se le asigna un `0`. Introducir ceros en conteos de palabras sesgará por completo las medidas de variabilidad (bajará artificialmente la media y aumentará falsamente la desviación estándar). Lo correcto para el análisis estadístico es usar valores nulos (`np.nan`), de modo que Pandas los ignore legítimamente al calcular las dispersiones.

En el código de abajo incluyo esta conversión automática.

---

### 1. Las Medidas Estadísticas Clave

Para medir la variabilidad de datos de texto (que suelen ser asimétricos), se debe calcular:

* **Mediana ($Q_2$):** El centro real de los datos sin verse afectado por empresas que escriben reportes masivos.
* **Desviación Estándar ($\sigma$):** La medida de dispersión clásica.
* **Rango Intercuartílico ($IQR = Q_3 - Q_1$):** Mide el ancho del 50% central de las empresas. Es una medida de variabilidad muy robusta frente a valores extremos (outliers).
* **Coeficiente de Variación ($CV = \frac{\sigma}{\mu}$):** **Esta es la métrica más importante para ti**. Divide la desviación estándar entre la media ($\mu$). Al ser un porcentaje/proporción adimensional, te permite comparar directamente la variabilidad de un ítem corto con uno largo.

---

### 2. Código en Python: Tabla de Métricas y Gráficas de Variabilidad



```python
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. PREPARACIÓN DE DATOS Y MÉTRICAS
# ==========================================

# Identificar las columnas de los ítems (todas menos 'Ticker')
item_columns = [col for col in df_item_comparison.columns if col != 'Ticker']

# Clonamos el dataframe y reemplazamos los 0 por NaN para no adulterar la variabilidad
df_stats = df_item_comparison.copy()
df_stats[item_columns] = df_stats[item_columns].replace(0, np.nan)

# Calculamos los estadísticos básicos transpuestos (.T)
summary_table = df_stats[item_columns].describe().T

# Añadimos el Rango Intercuartílico (IQR) y el Coeficiente de Variación (CV)
summary_table['IQR'] = summary_table['75%'] - summary_table['25%']
summary_table['CV'] = summary_table['std'] / summary_table['mean']

# Renombramos columnas para mayor claridad en español
summary_table = summary_table.rename(columns={
    'mean': 'Media',
    '50%': 'Mediana',
    'std': 'Desv. Estándar',
    'min': 'Mínimo',
    'max': 'Máximo'
})

print("=== TABLA RESUMEN DE VARIABILIDAD POR ÍTEM ===")
display(summary_table[['Media', 'Mediana', 'Desv. Estándar', 'IQR', 'CV', 'Mínimo', 'Máximo']])


# ==========================================
# 2. VISUALIZACIÓN GRÁFICA
# ==========================================

# Configuración estética general
sns.set_theme(style="whitegrid")

# --- GRÁFICA A: Boxplot (Diagrama de Caja y Bigotes) ---
# Transformamos a formato "largo" (melt) requerido por Seaborn para comparar variables
df_long = df_stats.melt(id_vars=['Ticker'], value_vars=item_columns, var_name='Item', value_name='Palabras')
df_long = df_long.dropna() # eliminamos nulos para graficar

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_long, x='Item', y='Palabras', palette='Set2', hue='Item', legend=False)
plt.yscale('log') # CRÍTICO: Escala logarítmica para absorber diferencias de tamaño
plt.title('Distribución y Variabilidad Absoluta del Conteo de Palabras por Ítem', fontsize=14, pad=15)
plt.xlabel('Sección del 10-K', fontsize=12)
plt.ylabel('Cantidad de Palabras (Escala Logarítmica)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- GRÁFICA B: Gráfico de Barras del Coeficiente de Variación (CV) ---
# Ordenamos los ítems de mayor a menor variabilidad relativa
summary_sorted = summary_table.sort_values(by='CV', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=summary_sorted.index, y=summary_sorted['CV'], palette='Reds_r', hue=summary_sorted.index, legend=False)
plt.title('Variabilidad Relativa entre Empresas (Coeficiente de Variación)\nA barra más alta, mayor heterogeneidad en el comportamiento de las empresas', fontsize=13, pad=15)
plt.xlabel('Sección del 10-K', fontsize=12)
plt.ylabel('Coeficiente de Variación (Desv. Std / Media)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- GRÁFICA C: Histograma de Densidad (KDE) para Ítems Clave ---
# Elegimos 3 ítems icónicos para ver la "forma" de su variabilidad (puedes cambiarlos)
items_interes = [col for col in ['Item 1', 'Item 1A', 'Item 7'] if col in item_columns]

if items_interes:
    plt.figure(figsize=(12, 5))
    for item in items_interes:
        sns.kdeplot(df_stats[item], label=item, fill=True, alpha=0.2)
    plt.title('Curva de Densidad: ¿Cómo se distribuye la longitud de los textos?', fontsize=14, pad=15)
    plt.xlabel('Cantidad de Palabras', fontsize=12)
    plt.ylabel('Densidad de Empresas', fontsize=12)
    plt.xlim(0, df_stats[items_interes].quantile(0.95).max()) # Limitamos al percentil 95 para que los outliers no deformen la gráfica
    plt.legend()
    plt.tight_layout()
    plt.show()

```



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. PREPARACIÓN DE DATOS Y MÉTRICAS
# ==========================================

# Identificar las columnas de los ítems (todas menos 'Ticker')
item_columns = [col for col in df_item_comparison.columns if col != 'Ticker']

# Clonamos el dataframe y reemplazamos los 0 por NaN para no adulterar la variabilidad
df_stats = df_item_comparison.copy()
df_stats[item_columns] = df_stats[item_columns].replace(0, np.nan)

# Calculamos los estadísticos básicos transpuestos (.T)
summary_table = df_stats[item_columns].describe().T

# Añadimos el Rango Intercuartílico (IQR) y el Coeficiente de Variación (CV)
summary_table['IQR'] = summary_table['75%'] - summary_table['25%']
summary_table['CV'] = summary_table['std'] / summary_table['mean']

# Renombramos columnas para mayor claridad en español
summary_table = summary_table.rename(columns={
    'mean': 'Media',
    '50%': 'Mediana',
    'std': 'Desv. Estándar',
    'min': 'Mínimo',
    'max': 'Máximo'
})

print("=== TABLA RESUMEN DE VARIABILIDAD POR ÍTEM ===")
display(summary_table[['Media', 'Mediana', 'Desv. Estándar', 'IQR', 'CV', 'Mínimo', 'Máximo']])


# ==========================================
# 2. VISUALIZACIÓN GRÁFICA
# ==========================================

# Configuración estética general
sns.set_theme(style="whitegrid")

# --- GRÁFICA A: Boxplot (Diagrama de Caja y Bigotes) ---
# Transformamos a formato "largo" (melt) requerido por Seaborn para comparar variables
df_long = df_stats.melt(id_vars=['Ticker'], value_vars=item_columns, var_name='Item', value_name='Palabras')
df_long = df_long.dropna() # eliminamos nulos para graficar

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_long, x='Item', y='Palabras', palette='Set2', hue='Item', legend=False)
plt.yscale('log') # CRÍTICO: Escala logarítmica para absorber diferencias de tamaño
plt.title('Distribución y Variabilidad Absoluta del Conteo de Palabras por Ítem', fontsize=14, pad=15)
plt.xlabel('Sección del 10-K', fontsize=12)
plt.ylabel('Cantidad de Palabras (Escala Logarítmica)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- GRÁFICA B: Gráfico de Barras del Coeficiente de Variación (CV) ---
# Ordenamos los ítems de mayor a menor variabilidad relativa
summary_sorted = summary_table.sort_values(by='CV', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=summary_sorted.index, y=summary_sorted['CV'], palette='Reds_r', hue=summary_sorted.index, legend=False)
plt.title('Variabilidad Relativa entre Empresas (Coeficiente de Variación)\nA barra más alta, mayor heterogeneidad en el comportamiento de las empresas', fontsize=13, pad=15)
plt.xlabel('Sección del 10-K', fontsize=12)
plt.ylabel('Coeficiente de Variación (Desv. Std / Media)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# --- GRÁFICA C: Histograma de Densidad (KDE) para Ítems Clave ---
# Elegimos 3 ítems icónicos para ver la "forma" de su variabilidad (puedes cambiarlos)
items_interes = [col for col in ['Item 1', 'Item 1A', 'Item 7'] if col in item_columns]

#items_interes = [col for col in tenk.items if col in item_columns]

if items_interes:
    plt.figure(figsize=(12, 5))
    for item in items_interes:
        sns.kdeplot(df_stats[item], label=item, fill=True, alpha=0.2)
    plt.title('Curva de Densidad: ¿Cómo se distribuye la longitud de los textos?', fontsize=14, pad=15)
    plt.xlabel('Cantidad de Palabras', fontsize=12)
    plt.ylabel('Densidad de Empresas', fontsize=12)
    plt.xlim(0, df_stats[items_interes].quantile(0.95).max()) # Limitamos al percentil 95 para que los outliers no deformen la gráfica
    plt.legend()
    plt.tight_layout()
    plt.show()

---

### 3. ¿Cómo interpretar las gráficas?

1. **El Boxplot (Gráfica A):** Te mostrará los límites de longitud de cada sección. El tamaño de la "caja" representa el $IQR$. Si un ítem tiene una caja muy larga, significa que las empresas escriben extensiones muy distintas. Los puntos sueltos arriba y abajo son *outliers*: empresas hiper-extensas o reportes inusualmente breves.
2. **El Gráfico del Coeficiente de Variación (Gráfica B):** Es tu métrica estandarizada. Si el *Item 1A (Riesgos)* muestra un $CV$ de $0.60$ y el *Item 7 (MD&A)* muestra un $CV$ de $0.30$, significa que **el tamaño de los factores de riesgo es el doble de inestable/variable** entre empresas que el análisis financiero, sin importar cuál de las dos secciones sea más larga en promedio.
3. **La Curva de Densidad (Gráfica C):** Te enseñará la "silueta" de los datos. Verás si la variabilidad se distribuye de forma simétrica o si tiene una gran asimetría hacia la derecha (muchas empresas concentradas en textos cortos y unas pocas firmas gigantescas estirando el gráfico hacia palabras infinitas).

### Visualizing Word Counts by Key Item and Ticker

This grouped bar chart visually compares the word counts of 'Item 1' (Business), 'Item 1A' (Risk Factors), and 'Item 7' (MD&A) across the selected tech companies. It helps in understanding which companies provide more detailed disclosures in these critical sections.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Melt the DataFrame to long format for easier plotting with seaborn
df_melted_plot = df_item_comparison.melt(
    id_vars=['Ticker'],
    value_vars=key_items_to_plot,
    var_name='Section Item',
    value_name='Word Count'
)

# Set up the plot style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 8))

# Create the grouped bar plot
sns.barplot(data=df_melted_plot, x='Section Item', y='Word Count', hue='Ticker', palette='viridis')

plt.title('Word Count Comparison for Key 10-K Sections by Ticker', fontsize=16)
plt.xlabel('10-K Section Item', fontsize=12)
plt.ylabel('Number of Words', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10) # Rotate labels for better readability
plt.yticks(fontsize=10)
plt.legend(title='Company Ticker', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, title_fontsize=12)
plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

To sort items like 'Item 1', 'Item 1A', 'Item 2', and 'Item 10' correctly (i.e., 'Item 1', 'Item 1A', 'Item 2', ..., 'Item 10'), a simple alphabetical sort isn't sufficient. Python's default `sort()` or `sorted()` would place 'Item 10' before 'Item 2' because it compares character by character. We need a custom sorting key that can extract the numerical part of the item and sort based on that, while also handling any trailing letters (like 'A' in 'Item 1A').

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# df_item_comparison and key_items_to_plot are available from previous execution

# Define the items for each part of the 10-K based on the provided structure
parts_definition = {
    "Parte I: El Corazón del Negocio y sus Riesgos": [
        'Item 1', 'Item 1A', 'Item 1B', 'Item 1C', 'Item 2', 'Item 3', 'Item 4'
    ],
    "Parte II: Los Números y el Análisis Financiero": [
        'Item 5', 'Item 6', 'Item 7', 'Item 7A', 'Item 8', 'Item 9', 'Item 9A', 'Item 9B', 'Item 9C'
    ],
    "Parte III: Gobernanza y Compensaciones": [
        'Item 10', 'Item 11', 'Item 12', 'Item 13', 'Item 14'
    ],
    "Parte IV: Anexos y Firmas": [
        'Item 15', 'Item 16'
    ]
}

sns.set_theme(style="whitegrid")

for part_name, items_in_part in parts_definition.items():
    # Filter df_item_comparison for items relevant to the current part
    # Ensure that the items_in_part actually exist in the DataFrame columns
    valid_items_for_plot = [item for item in items_in_part if item in df_item_comparison.columns]

    if not valid_items_for_plot:
        print(f"No data available for {part_name}. Skipping plot.")
        continue

    # Select only the 'Ticker' column and the valid items for this part
    df_part = df_item_comparison[['Ticker'] + valid_items_for_plot]

    # Melt the DataFrame to long format for easier plotting with seaborn
    df_melted_part = df_part.melt(
        id_vars=['Ticker'],
        value_vars=valid_items_for_plot,
        var_name='Sección del 10-K',
        value_name='Número de Palabras'
    )

    plt.figure(figsize=(14, 8))
    sns.barplot(data=df_melted_part, x='Sección del 10-K', y='Número de Palabras', hue='Ticker', palette='viridis')

    plt.title(f'Comparación del Número de Palabras por Ticker: {part_name}', fontsize=16)
    plt.xlabel('Sección del 10-K', fontsize=12)
    plt.ylabel('Número de Palabras', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(fontsize=10)
    plt.legend(title='Ticker de la Empresa', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, title_fontsize=12)
    plt.tight_layout()
    plt.show()

## All Available Sections

A 10-K has over 20 sections. See what's available and how much text each contains:

In [ ]:
for item in tenk.items:
    text = tenk[item]
    words = len(text.split())
    print(f"{item:12s} {words:>8,} words  {len(text):>8,} chars")
    print(text)

## Search Within a Filing

Search for specific terms or topics within a filing to find relevant passages:

In [ ]:
results = filing.search("artificial intelligence")
print(f"Found {len(results)} matching sections")
results

## Structured Chunks for LLM Context

The chunked document breaks a filing into labeled segments, ideal for RAG pipelines or fitting within LLM context windows:

In [ ]:
chunked = tenk.chunked_document
df = chunked.as_dataframe()

print(f"Total chunks: {len(df)}")
print(f"Columns: {df.columns.tolist()}\n")

# Chunks per section
sections = df[df["Item"] != ""].groupby("Item").agg(
    Chunks=("Chars", "count"),
    Total_Chars=("Chars", "sum")
).sort_values("Total_Chars", ascending=False)

sections.head(10)

## Compare Text Volumes Across Companies

Different companies disclose varying levels of detail. Compare text volumes to understand disclosure depth:

In [ ]:
tickers = ["NVDA", "MSFT", "AAPL", "GOOG"]
rows = []

for ticker in tickers:
    filing = Company(ticker).get_filings(form="10-K")[0]
    tenk = filing.obj()
    text = filing.text()
    rows.append({
        "Ticker": ticker,
        "Total Words": f"{len(text.split()):,}",
        "Business": f"{len(tenk['1'].split()):,}",
        "Risk Factors": f"{len(tenk['1A'].split()):,}",
        "MD&A": f"{len(tenk['7'].split()):,}",
    })

pd.DataFrame(rows).set_index("Ticker")

## Why EdgarTools?

EdgarTools is free and open-source. Compare extracting SEC filing text:

**With edgartools (free, no API key):**
```python
filing = Company("NVDA").get_filings(form="10-K")[0]
text = filing.text()            # Full text
tenk = filing.obj()
tenk["1A"]                      # Risk factors section
filing.search("AI")             # Search within filing
```

**Typical paid API approach ($50+/month, API key required):**
```python
from sec_api import ExtractorApi
api = ExtractorApi(api_key="YOUR_PAID_API_KEY")
text = api.get_section(url, "1A", "text")  # One section per API call
# ... rate-limited, paid per request, no search capability
```

With edgartools, the entire filing is parsed locally -- all sections, search, and chunks available instantly with no per-request cost.

## Quick Reference

```python
from edgar import *
set_identity("your.name@example.com")

# ── Full filing text ──
filing = Company("NVDA").get_filings(form="10-K")[0]
filing.text()                          # Plain text
filing.markdown()                      # Markdown
filing.html()                          # HTML

# ── Sections by item number ──
tenk = filing.obj()
tenk["1"]                              # Business description
tenk["1A"]                             # Risk factors
tenk["7"]                              # MD&A
tenk.items                             # List all items

# ── Search ──
filing.search("artificial intelligence")  # Find matching passages

# ── Structured chunks ──
chunked = tenk.chunked_document
df = chunked.as_dataframe()            # All chunks with metadata
# Columns: Text, Table, Chars, Part, Item
```

## What's Next

You've learned how to extract SEC filing text for NLP analysis. Here are related tutorials:

- [Analyze 10-K Annual Reports](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/analyze-10k-annual-report-python.ipynb)
- [Download SEC Filings in Bulk](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/download-sec-filings-bulk-python.ipynb)
- [Search and Filter SEC Filings](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/search-sec-filings-python.ipynb)
- [SEC EDGAR API in Python](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/sec-edgar-api-python.ipynb)

**Resources:**
- [EdgarTools Documentation](https://edgartools.readthedocs.io/)
- [GitHub Repository](https://github.com/dgunning/edgartools)
- [PyPI Package](https://pypi.org/project/edgartools/)

---

## Support EdgarTools

If you found this tutorial helpful, here are a few ways to support the project:

- **Star the repo** -- [github.com/dgunning/edgartools](https://github.com/dgunning/edgartools) -- it helps others discover edgartools
- **Visit edgartools.io** -- [edgartools.io](https://www.edgartools.io/) -- for more tutorials, articles, and updates
- **Report issues** -- found a bug or have a feature idea? [Open an issue](https://github.com/dgunning/edgartools/issues)
- **Share this notebook** -- know someone who works with SEC data? Send them the Colab link

*edgartools is free, open-source, and community-driven. No API key or paid subscription required.*